# Fine-Tune XTTS On Shona In Colab

This notebook is designed for free Colab GPU use.

It supports two repo-loading paths:
- clone the repo from GitHub
- upload `cloner.zip` directly from your machine into Colab


## Outline

1. Check GPU runtime
2. Optionally mount Drive for checkpoints
3. Load the repo into `/content/cloner`
4. Install dependencies
5. Build the filtered Shona dataset from Hugging Face
6. Start XTTS fine-tuning
7. Resume from checkpoints later


In [1]:
# Step 1: verify that Colab gave you a GPU runtime.
!nvidia-smi || true

import torch
HAS_CUDA = torch.cuda.is_available()
print('cuda:', HAS_CUDA)
if HAS_CUDA:
    print('device:', torch.cuda.get_device_name(0))
else:
    print('No CUDA GPU runtime is attached.')
    print('In Colab use Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')


Wed Jul  8 08:41:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

cuda: True
device: Tesla T4


In [ ]:
# Step 2: mount Drive only if you want checkpoints to persist automatically.
USE_DRIVE = False

PROJECT_ROOT = '/content/cloner'
CHECKPOINT_DIR = '/content/checkpoints/shona_xtts'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DRIVE_ROOT = '/content/drive/MyDrive/shona_xtts'
    CHECKPOINT_DIR = f'{DRIVE_ROOT}/checkpoints/shona_xtts'

print('CHECKPOINT_DIR =', CHECKPOINT_DIR)


CHECKPOINT_DIR = /content/checkpoints/shona_xtts


## Step 3A - Clone From GitHub

Use this to clone the repo into Colab.


In [3]:
# Step 3A: clone from GitHub.
# Replace the URL before running.
GIT_REPO_URL = 'https://github.com/Rakinzi/cloner'

import os, shutil
if GIT_REPO_URL:
    if os.path.exists(PROJECT_ROOT):
        shutil.rmtree(PROJECT_ROOT)
    !git clone "$GIT_REPO_URL" "$PROJECT_ROOT"
    !ls -la "$PROJECT_ROOT"
else:
    print('Set GIT_REPO_URL before running this cell.')


Cloning into '/content/cloner'...
remote: Enumerating objects: 33, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (29/29), done.
remote: Total 33 (delta 1), reused 33 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (33/33), 231.57 KiB | 1.24 MiB/s, done.
Resolving deltas: 100% (1/1), done.
total 684
drwxr-xr-x 6 root root   4096 Jul  8 08:41 .
drwxr-xr-x 1 root root   4096 Jul  8 08:41 ..
drwxr-xr-x 5 root root   4096 Jul  8 08:41 app
-rw-r--r-- 1 root root      0 Jul  8 08:41 app.py
-rw-r--r-- 1 root root    287 Jul  8 08:41 docker-compose.yml
-rw-r--r-- 1 root root    406 Jul  8 08:41 Dockerfile
drwxr-xr-x 8 root root   4096 Jul  8 08:41 .git
-rw-r--r-- 1 root root    607 Jul  8 08:41 .gitignore
drwxr-xr-x 3 root root   4096 Jul  8 08:41 output
-rw-r--r-- 1 root root    669 Jul  8 08:41 pyproject.toml
-rw-r--r-- 1 root root    163 Jul  8 08:41 requirements.txt
drwxr-xr-x 2 root root   4096 Jul  8 08:41 scripts
-rw-r--r-- 1 root root   

In [4]:
# Step 3C: verify the repo is present before continuing.
import os
assert os.path.exists(PROJECT_ROOT), 'Repo missing. Run Step 3A first.'
assert os.path.exists(f'{PROJECT_ROOT}/pyproject.toml'), 'pyproject.toml missing. The repo did not extract correctly.'
print('Repo is ready at', PROJECT_ROOT)


Repo is ready at /content/cloner


In [5]:
# Step 4: install dependencies.
import os
assert os.path.exists(f'{PROJECT_ROOT}/pyproject.toml'), 'Repo missing. Run Step 3A first.'
%cd /content/cloner
!python -V
!pip install -U pip setuptools wheel
!pip install uv
!pip install -U torchcodec
!uv sync


/content/cloner
Python 3.12.13
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 134 packages in 2ms
Installed 125 packages in 509ms                             
 + absl-py==2.4.0
 + aiofiles==25.1.0
 + aiohappyeyeballs==2.6.2
 + aiohttp==3.14.1
 + aiosignal==1.4.0
 + annotated-types==0.7.0
 + anyascii==0.3.3
 + anyio==4.13.0
 + attrs==26.1.0
 + audioread==3.1.0
 + certifi==2026.5.20
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.4.1
 + contourpy==1.3.3
 + coqpit-config==0.2.5
 + coqui-tts==0.27.5
 + coqui-tts-trainer==0.3.3
 + cutlet==0.5.0
 + cycler==0.12.1
 + datasets==5.0.0
 + decorator==5.3.1
 + dill==0.4.1
 + docopt==0.6.2
 + einops==0.8.2
 + fastapi==0.115.6
 + filelock==3.29.3
 + fonttools==4.63.0
 + frozenlist==1.8.0
 + fsspec==2026.4.0
 + fugashi==1.5.2
 + grpcio==1.81.1
 + h11==0.16.0
 + hf-xet==1.5.1
 + httpcore==1.0.9
 + httptools==0.8.0
 + httpx==0.28.1
 + huggingface-hub==0.36.2
 + idna==3.18
 + inflect==7.5.0

In [6]:
# Step 5: build the filtered XTTS dataset directly from Hugging Face.
import os, shutil
assert os.path.exists(f'{PROJECT_ROOT}/scripts/build_hf_xtts_dataset.py'), 'Missing build_hf_xtts_dataset.py. Repo not loaded correctly.'

HF_DATASET = 'manassehzw/sna-dataset-annotated'
TARGET_DATASET_DIR = f'{DRIVE_ROOT}/sna_xtts_ft_filtered' if 'DRIVE_ROOT' in globals() else '/content/sna_xtts_ft_filtered'

print('HF_DATASET =', HF_DATASET)
print('TARGET_DATASET_DIR =', TARGET_DATASET_DIR)
if os.path.exists(TARGET_DATASET_DIR):
    shutil.rmtree(TARGET_DATASET_DIR)

%cd /content/cloner
!uv run python scripts/build_hf_xtts_dataset.py \
  --output "$TARGET_DATASET_DIR" \
  --min-quality 70 \
  --min-duration 3 \
  --max-duration 20 \
  --max-clips-per-speaker 200

!find "$TARGET_DATASET_DIR/wavs" -type f -name '*.wav' | wc -l
!du -sh "$TARGET_DATASET_DIR"


AssertionError: Source dataset not found: /Volumes/LaCie/WaxalNLP/sna_asr

In [ ]:
# Step 6: launch XTTS fine-tuning.
# The dataset is Shona. `--xtts-language en` is only the XTTS-supported language token workaround.
if not HAS_CUDA:
    raise RuntimeError('Stop here until Colab gives you a GPU runtime. Rerun Step 1 after changing runtime type.')

assert os.path.exists(TARGET_DATASET_DIR), f'Dataset not found: {TARGET_DATASET_DIR}'
%cd /content/cloner
!MPLBACKEND=Agg uv run python scripts/finetune_xtts.py \
  --dataset "$TARGET_DATASET_DIR" \
  --output "$CHECKPOINT_DIR" \
  --xtts-language en \
  --batch-size 1 \
  --grad-accum 84 \
  --steps 5000 \
  --save-step 500 \
  --workers 2


In [ ]:
# Step 7: resume later if Colab disconnects.
LATEST_CHECKPOINT = ''

if LATEST_CHECKPOINT:
    assert os.path.exists(TARGET_DATASET_DIR), f'Dataset not found: {TARGET_DATASET_DIR}'
    %cd /content/cloner
    !MPLBACKEND=Agg uv run python scripts/finetune_xtts.py \
      --dataset "$TARGET_DATASET_DIR" \
      --output "$CHECKPOINT_DIR" \
      --xtts-language en \
      --resume "$LATEST_CHECKPOINT" \
      --batch-size 1 \
      --grad-accum 84 \
      --steps 5000 \
      --save-step 500 \
      --workers 2
else:
    print('Set LATEST_CHECKPOINT to resume from an interrupted run.')


## Notes

- If Google Drive auth is flaky, keep `USE_DRIVE = False` and train under `/content`.
- If the Colab session resets, rerun all cells after Step 1.
- XTTS still uses `en` as the supported internal language token even though the dataset is Shona.
